# 1. Financial sentiment: data and honest baselines

**Goal:** understand our labels, build reproducible splits, and evaluate a simple text classifier before using an 8-billion-parameter model. No GPU, Hugging Face token, or paid service is needed for this lesson.

We use `zeroshot/twitter-financial-news-sentiment`: 0 = bearish, 1 = bullish, 2 = neutral. These are sentiment annotations, not future returns. The publisher declares MIT licensing. Data stays out of Git.

Open this notebook in Google Colab and run cells from top to bottom. The first code cell downloads the public tutorial repository automatically.

[Open this lesson in Colab](https://colab.research.google.com/github/chenghungpan/llama3-qlora-finance-colab/blob/main/notebooks/01_data_and_baselines.ipynb) · [GitHub repository](https://github.com/chenghungpan/llama3-qlora-finance-colab)


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

# The public repository is downloaded automatically. Clear this URL to upload a project ZIP instead.
REPO_URL = "https://github.com/chenghungpan/llama3-qlora-finance-colab.git"
PROJECT_NAME = "llama3-qlora-finance-colab"
if Path("finance_qlora/core.py").exists():
    ROOT = Path.cwd()
else:
    parent = Path("/content") if Path("/content").exists() else Path.cwd()
    ROOT = parent / PROJECT_NAME
    if not ROOT.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
        else:
            from google.colab import files
            print("Upload llama3-qlora-finance-colab.zip (the whole project, not just a notebook).")
            uploaded = files.upload()
            archive = next((name for name in uploaded if name.endswith(".zip")), None)
            if archive is None:
                raise ValueError("Please upload the project ZIP")
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (parent / member.filename).resolve()
                    if not target.is_relative_to(parent.resolve()):
                        raise ValueError("Unsafe archive path")
                z.extractall(parent)
    if not (ROOT / "finance_qlora/core.py").exists():
        raise ValueError("Expected the standalone project directory in the ZIP")
    os.chdir(ROOT)
print("Project directory:", ROOT)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cpu.txt", "-e", "."], check=True)


## Download and audit

The downloader uses two CSV files at a pinned commit; it never runs remote Python code. We preserve upstream validation as our final **test** set. We take a new stratified validation split from upstream training.

Before splitting, we normalize whitespace/case, drop conflicting-label duplicate groups, and prevent exact duplicates from crossing boundaries. For a duplicate shared by upstream train and test, we keep the test copy and remove the training copies. Semantic near-duplicates may remain.


In [ ]:
subprocess.run([sys.executable, "-m", "finance_qlora.data"], check=True)


In [ ]:
import json
from collections import Counter
from finance_qlora.core import read_jsonl, LABELS
train = read_jsonl("data/train.jsonl")
validation = read_jsonl("data/validation.jsonl")
for name, rows in [("train", train), ("validation", validation)]:
    print(name, len(rows), Counter(LABELS[r["label"]] for r in rows))
# Only inspect training examples while designing the experiment.
for row in train[:5]:
    print(LABELS[row["label"]], "|", row["text"])


## Compare cheap baselines on validation

A majority classifier predicts the most frequent training class. TF-IDF converts words and word pairs into weighted features; logistic regression learns a decision boundary. Fit vocabulary and weights on training data only.

Accuracy is the fraction correct. Macro-F1 averages each class's F1 equally, preventing a large neutral class from dominating the score. Here we use validation so the final test remains available for the fixed experiment in lesson 2.


In [ ]:
from finance_qlora.baselines import evaluate_baselines
scores = evaluate_baselines(train, validation)
print(json.dumps(scores, indent=2))


## Exercise

Which class has the lowest recall? Inspect ten misclassifications from a classifier you fit using this module's code. Consider ambiguous language, sarcasm, multiple companies, and time context. Do not rewrite test labels to make the model look better.

Next: open `02_qlora_training.ipynb`. You may use a new Colab runtime; it can reproduce the same data from scratch.
